In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve the secret
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_Token")

# Log in
login(token=hf_token)

In [3]:
!pip install -U bitsandbytes peft
!install transformers==4.38.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.5 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
install: missing destination file operand after 'transformers==4.38.0'
Try 'install --help' for more information.


In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/adapter_model.safetensors
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/trainer_state.json
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/training_args.bin
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/adapter_config.json
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/README.md
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/tokenizer.json
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/tokenizer_config.json
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/chat_template.jinja
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/scheduler.pt
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/optimizer.pt
/kaggle/input/datasets/bhavyranka/my-model-checkpoint-1000-zip/rng_state.pth
/kaggle/input/datasets/bhavyranka/checkpoint-1200-backup-zip/adapter_model.safetensors
/kaggle/input/datasets/bhavyra

In [7]:
import os
import json
import random
import logging
 
import numpy as np
import pandas as pd
import torch
 
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from datasets import Dataset


In [8]:
folder_path = "/kaggle/input/datasets/bhavyranka/fd-llm-crwu-dataset"
# FOLDER_PATH   = "/kaggle/input/datasets/bhavyranka/dataset"
DATA_FILENAME = "cwru_stat_dataset.json"          # or cwru_fft_dataset.json
DATA_TYPE     = "stat"                             # "stat" | "fft"
OUTPUT_DIR    = "/kaggle/working/fd_llm_output"
 
MODEL_ID      = "meta-llama/Meta-Llama-3-8B-Instruct"
LOAD_IN_4BIT  = True
 
LORA_RANK     = 4
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
 
LEARNING_RATE = 1e-4
BATCH_SIZE    = 2
NUM_EPOCHS    = 4
MAX_SEQ_LEN   = 1024
SEED          = 42
TEST_SIZE     = 0.10
 
# Set to a path string to skip training and load a checkpoint instead
EVAL_ONLY       = False
CHECKPOINT_DIR  = "/kaggle/working/resumed_checkpoint"
 
CROSS_EVAL_PATH = None

In [9]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)
 
FAULT_LABELS = ["NO", "IRF", "ORF", "REF"]
LABEL2ID     = {l: i for i, l in enumerate(FAULT_LABELS)}
ID2LABEL     = {i: l for l, i in LABEL2ID.items()}
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [10]:
def build_prompt(instruction: str, input_text: str, output: str = "") -> str:
    """Alpaca-style prompt. output omitted for inference."""
    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )
    if output:
        prompt += output
    return prompt
 
def build_inference_prompt(instruction: str, input_text: str) -> str:
    return build_prompt(instruction, input_text, output="")


In [11]:
data_path = os.path.join(folder_path, DATA_FILENAME)
 
with open(data_path, "r") as f:
    data = json.load(f)
logger.info(f"Loaded {len(data)} samples from {data_path}")
 
# Label distribution
from collections import Counter
logger.info(f"Label distribution [full]: {dict(Counter(d['output'] for d in data))}")
 
unknown_labels = {d["output"] for d in data if d["output"] not in LABEL2ID}
if unknown_labels:
    logger.warning(f"Unknown labels found: {unknown_labels}")
 
train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[d["output"] for d in data]
)
logger.info(f"Train: {len(train_data)} | Test: {len(test_data)}")
logger.info(f"Label distribution [train]: {dict(Counter(d['output'] for d in train_data))}")
logger.info(f"Label distribution [test]:  {dict(Counter(d['output'] for d in test_data))}")


03:36:24 | INFO | Loaded 3692 samples from /kaggle/input/datasets/bhavyranka/fd-llm-crwu-dataset/cwru_stat_dataset.json
03:36:24 | INFO | Label distribution [full]: {'NO': 1034, 'IRF': 886, 'ORF': 887, 'REF': 885}
03:36:24 | INFO | Train: 3322 | Test: 370
03:36:24 | INFO | Label distribution [train]: {'ORF': 798, 'REF': 796, 'NO': 931, 'IRF': 797}
03:36:24 | INFO | Label distribution [test]:  {'IRF': 89, 'NO': 103, 'REF': 89, 'ORF': 89}


In [12]:
def tokenize_sample(sample, tokenizer, max_seq_len: int):
    full_text   = build_prompt(sample["instruction"], sample["input"], sample["output"])
    prompt_only = build_inference_prompt(sample["instruction"], sample["input"])
 
    full_ids   = tokenizer(full_text,   truncation=True, max_length=max_seq_len)["input_ids"]
    prompt_ids = tokenizer(prompt_only, truncation=True, max_length=max_seq_len)["input_ids"]
 
    # Mask prompt tokens from the loss
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
 
    return {
        "input_ids":      full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels":         labels,
    }
 
def make_hf_dataset(data, tokenizer, max_seq_len: int) -> Dataset:
    tokenized = [tokenize_sample(s, tokenizer, max_seq_len) for s in data]
    return Dataset.from_list(tokenized)


In [13]:
logger.info(f"Loading tokenizer and model: {MODEL_ID}")
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
 
bnb_config = None
if LOAD_IN_4BIT:
    try:
        import bitsandbytes
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        logger.info("4-bit quantisation enabled (bitsandbytes).")
    except ImportError:
        logger.warning("bitsandbytes not found — loading in bfloat16 instead.")
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if bnb_config is None else None,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False


03:36:24 | INFO | Loading tokenizer and model: meta-llama/Meta-Llama-3-8B-Instruct
03:36:24 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
03:36:24 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

03:36:24 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
03:36:24 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

03:36:24 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
03:36:24 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
03:36:24 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
03:36:24 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
03:36:24 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


...

03:36:31 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00001-of-00004.safetensors "HTTP/1.1 302 Found"
03:36:31 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00003-of-00004.safetensors "HTTP/1.1 302 Found"
03:36:31 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00004-of-00004.safetensors "HTTP/1.1 302 Found"
03:36:31 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/xet-read-token/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2 "HTTP/1.1 200 OK"
03:36:31 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/xet-read-token/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2 "HTTP/1.1 200 OK"
03:36:31 | INFO | HTTP Request: GET https:/

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

03:38:11 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
03:38:11 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

03:38:11 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


In [14]:
if EVAL_ONLY:
    _ckpt = CHECKPOINT_DIR or os.path.join(OUTPUT_DIR, "checkpoint-final")
    logger.info(f"Eval-only mode: loading LoRA checkpoint from {_ckpt}")
    model = PeftModel.from_pretrained(model, _ckpt)
else:
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "v_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()


trainable params: 1,703,936 || all params: 8,031,965,184 || trainable%: 0.0212


In [16]:
if not EVAL_ONLY:
    train_ds = make_hf_dataset(train_data, tokenizer, MAX_SEQ_LEN)
    eval_ds  = make_hf_dataset(test_data,  tokenizer, MAX_SEQ_LEN)
 
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
        gradient_accumulation_steps=4,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,          # keep last 3 checkpoints to save disk
        eval_strategy="steps",
        eval_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=0,
    )
 
    data_collator = DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True, pad_to_multiple_of=8
    )
 
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
    )
 
    logger.info("Starting fine-tuning …")
    # trainer.train()
    trainer.train(resume_from_checkpoint="/kaggle/working/resumed_checkpoint_1200")
 
    # Save final checkpoint
    final_dir = os.path.join(OUTPUT_DIR, "checkpoint-final")
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    logger.info(f"Final model saved to {final_dir}")


03:39:03 | INFO | Starting fine-tuning …


Step,Training Loss,Validation Loss
1250,0.068794,0.048340
1300,0.045763,0.032246
1350,0.019150,0.030233
1400,0.038574,0.015942
1450,0.020076,0.014270
1500,0.023571,0.015090
1550,0.018957,0.016281
1600,0.058566,0.015183
1650,0.037301,0.015569


06:13:30 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
06:13:30 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
08:48:23 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
08:48:23 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
09:34:15 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
09:34:15 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
09:34:16 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
09:34:16 | INFO | HTTP Requ

In [6]:
import shutil

# This zips the entire checkpoint folder
# shutil.make_archive('my_model_checkpoint_1000', 'zip', '/kaggle/working/fd_llm_output/checkpoint-1000')

# print("Zip file created! Check the 'Output' sidebar to download.")

In [ ]:
!ls /kaggle/working/resumed_checkpoint

In [17]:
import shutil
import os

# Zip the specific checkpoint folder
checkpoint_path = '/kaggle/working/fd_llm_output/checkpoint-1664'
output_zip = '/kaggle/working/checkpoint_1664_backup'

shutil.make_archive(output_zip, 'zip', checkpoint_path)

print(f"Zip file created at: {output_zip}.zip")

Zip file created at: /kaggle/working/checkpoint_1664_backup.zip


In [ ]:
input_path = '/kaggle/input/datasets/bhavyranka/checkpoint-1200-backup-zip' 
!cp -r {input_path} /kaggle/working/resumed_checkpoint_1200